This demo shows how to make PFVs for Manwe, from dSv1

# Imports

In [ ]:
from cytools import Polytope

In [ ]:
from pfvs import CYData, PFV, pvecs, ZpM, coniZpM

In [ ]:
import numpy as np

In [ ]:
import pfvs

# Load Manwe

In [ ]:
verifying_from_dsV1_repo = False

## From the dSv1 data file (for verification)

In [ ]:
if verifying_from_dsV1_repo:
    import gzip, pickle

    # load the dataframe
    with gzip.open('dSv1.p', 'rb') as f:
        dSv1_df = pickle.load(f)

    # read the CY data
    manwe_df = dSv1_df[dSv1_df['name']=='manwe'].iloc[0]
    p  = Polytope(manwe_df['dual points'])
    t  = p.triangulate(heights=manwe_df['mirror heights'])
    cy = t.cy()

## Hard-coded Manwe's CY

In [ ]:
if not verifying_from_dsV1_repo:
    verts   = [[0, 0, 0, 0], [1, -1, -1, -1], [-1, 2, 1, 1], [-1, -1, 0, 0], [-1, -1, 2, 0], [-1, -1, 2, 1], [-1, 0, 0, 2], [-1, -1, 0, 2], [-1, 0, 0, 1], [-1, 0, 1, 0], [-1, -1, 0, 1], [-1, -1, 1, 0], [-1, -1, 1, 1], [-1, 0, 1, 1], [-1, 1, 1, 1], [0, -1, 0, 0]]
    heights = [0, 35, 29, 35, 31, 35, 35, 35, 15, 17, 31, 9, 21]
    p       = Polytope(verts)
    t       = p.triangulate(heights=heights)
    cy      = t.cy()

## Set the conifold-related info

In [ ]:
# Manwe's conifold charge (precomputed; the conifold-discovery step is
# omitted here so the demo is self-contained)
q = np.array([0, 0, 0, 0, 0, -1, 0, 0])

# set the COB (same one used in paper)
cob = np.array([[0,0,0,0,0,-1,0,0],
[-1,0,0,0,0,0,0,0],
[0,-1,0,0,0,0,0,0],
[0,0,-1,0,0,0,0,0],
[0,0,0,-1,0,0,0,0],
[0,0,0,0,-1,0,0,0],
[0,0,0,0,0,0,-1,0],
[0,0,0,0,0,0,0,-1]])

## Hard-code Manwe

In [ ]:
data = CYData.from_cy(cy, coni_curve=q, coni_cob=cob)

In [ ]:
ps = pvecs(data, 10_000)

In [ ]:
K = np.array([-6, -1,   0, 1, -3,  2,  0, -1])
M = np.array([16, 10, -26, 8, 32, 30, 18, 28])
manwe = PFV(data, K=K, M=M)
print(manwe.check_all())

Set GVs (for fancier diagnostics)

In [ ]:
manwe.gvs = manwe.cy.compute_gvs(max_deg=10)
manwe.diagnostics()

# Find Manwe from scratch

Get some p-vectors

In [ ]:
import time
tic = time.time()
ps = pvecs(data, min_N_pts=100_000)#, max_bytes=1e12)
toc = time.time()

In [ ]:
toc-tic

Do the search!

In [ ]:
results = coniZpM(
    data=data,
    ps=ps,
    Q=(data.h11+data.h21+4),
    M0min=13,
    ellipsoid_dilation=50,
    max_N_pfvs=100_000_000,
    return_formal_pfvs=True,
    verbosity=0,
    n_jobs=1,
)

In [ ]:
p = 3
results = coniZpM(
    data=data,
    ps=ps,
    Q=(data.h11+data.h21+2+2*p),
    M0min=13*p,
    ellipsoid_dilation=1000,
    max_N_pfvs=100_000_000,
    return_formal_pfvs=True,
    verbosity=0,
    n_jobs=-1,
)

In [ ]:
len(results)

In [ ]:
results

In [ ]:
np.argmax([pfv.M[0] for pfv in results])

In [ ]:
i = 1
results[i].gvs = results[i].cy.compute_gvs(max_deg=10)

In [ ]:
(1/(2*np.pi)) * (np.exp(-2*np.pi*(data.h11+data.h21+2+2*p)/(2*results[i].gs*results[i].M[0]**2)))

In [ ]:
results[i].W0()

In [ ]:
results[i].gs*results[i].M[0]

In [ ]:
Ms = [pfv.M[0] for pfv in results]

In [ ]:
import matplotlib.pyplot as plt
plt.hist(Ms)
plt.yscale('log')

In [ ]:
## for i, pfv in enumerate(results):
    if all(pfv.K == manwe.K) and all(pfv.M == manwe.M):
        print(f"we found manwe at index {i} :)")
        break

## Plot the search results

Assign GVs to all of the PFVs

In [ ]:
gvs = results[0].cy.compute_gvs(max_deg=10).coo

for pfv in results:
    pfv.gvs = gvs

Extract data of interest

In [ ]:
W0s    = [pfv.W0() for pfv in results]
aligns = [pfv.align for pfv in results]
gsMs   = [pfv.gsM for pfv in results]

Plot it!

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(W0s, aligns, c=gsMs, s=4)
plt.scatter([manwe.W0()], [manwe.align], c=[manwe.gsM], marker='*', s=100)

plt.colorbar(label='gsM')

# axis scaling
plt.xscale('log')
plt.yscale('log')

# axis labels
plt.title("PFVs from Manwe's conifold")
plt.xlabel('W0')
plt.ylabel('align')

#plt.xlim([None,1])
#plt.ylim([0.05,2])